# vLLM vs SGLang on a Colab GPU
Runtime → Change runtime type → **L4 GPU**, then Run all.

- Installs each engine in its own virtualenv (about 10 minutes).
- Runs a quick smoke test (about 10 minutes), then the full sweep (about 50 minutes).
- Total is roughly 1.5 hours, about 3 compute units. Background execution is fine.

The notebook **stops** (raises an error) if an engine fails to install or the smoke test fails, so a broken setup does not burn an hour. If that happens, send the output of the failing cell.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os
if not os.path.isdir('/content/lowbit-decode-gpu'):
    !git clone -q https://github.com/FredWen-S/lowbit-decode-gpu.git /content/lowbit-decode-gpu
%cd /content/lowbit-decode-gpu
!git pull -q && git log -1 --format='%h %s'

Each engine gets its own virtualenv, because their torch pins conflict. The load generator runs in Colab's own Python.

In [ ]:
!pip -q install uv
!uv venv --python 3.12 /content/venv-vllm
!uv pip install --python /content/venv-vllm/bin/python vllm 2>&1 | tail -25

In [ ]:
!uv venv --python 3.12 /content/venv-sglang
!uv pip install --python /content/venv-sglang/bin/python 'sglang[all]' 2>&1 | tail -25

In [ ]:
import subprocess
for eng in ['vllm', 'sglang']:
    r = subprocess.run([f'/content/venv-{eng}/bin/python', '-c', f'import {eng}; print({eng}.__version__)'],
                       capture_output=True, text=True)
    print(eng, r.stdout.strip() or r.stderr[-2000:])
    assert r.returncode == 0, f'{eng} did not install; see the install cell above'

In [ ]:
from huggingface_hub import snapshot_download
MODEL = snapshot_download('Qwen/Qwen2.5-1.5B-Instruct', local_dir='/content/models/qwen2.5-1.5b-instruct')
!pip -q install aiohttp
print(MODEL)

Smoke test: every config on both engines, with few requests. If a server fails to start, the last lines of its log are printed here.

In [ ]:
!rm -f /content/SMOKE_FAILED
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --quick --out results/serving_quick || touch /content/SMOKE_FAILED
import os
assert not os.path.exists('/content/SMOKE_FAILED'), 'smoke test failed; see the output above'

Full sweep.

In [ ]:
!python bench/serve_sweep.py --model /content/models/qwen2.5-1.5b-instruct --python vllm=/content/venv-vllm/bin/python --python sglang=/content/venv-sglang/bin/python --out results/serving

In [ ]:
!cd results && zip -qr /content/serving_results.zip serving serving_quick
from google.colab import files
files.download('/content/serving_results.zip')